In [ ]:
import os
import time
import copy
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    matthews_corrcoef,
    confusion_matrix,
    balanced_accuracy_score,
    brier_score_loss,
    roc_curve,
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")


# ============================================================
# GLOBAL VARIABLES
# ============================================================

# ---------- DATA ----------
TRAIN_PATH = "/content/train_selected.csv"
TEST_PATH  = "/content/test_selected.csv"
TARGET_COLUMN = "lung_cancer_risk"
ID_COLUMNS = []
DROP_COLUMNS = []

# ---------- REPRO ----------
SEED = 42

# ---------- CV ----------
OUTER_FOLDS = 5
INNER_FOLDS = 2

# ---------- SPEED / TRAINING ----------
N_TRIALS = 3
MAX_EPOCHS_INNER = 6
MAX_EPOCHS_OUTER = 10
MAX_EPOCHS_FINAL = 12

PATIENCE_INNER = 2
PATIENCE_OUTER = 3
PATIENCE_FINAL = 3

OUTER_DEV_VALID_SIZE = 0.12
FINAL_VALID_SIZE = 0.10

USE_AMP = True
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- THRESHOLD ----------
THRESHOLD_GRID = np.linspace(0.16, 0.60, 177)
ACC_DROP_TOL = 0.0015

# Dynamic precision protection
MIN_DYNAMIC_PRECISION = 0.970
PRECISION_RELAX = 0.012

# ---------- LOSS ----------
USE_MILD_POS_WEIGHT = True
POS_WEIGHT_POWER = 0.35
POS_WEIGHT_MAX = 1.25

# ---------- HP SELECTION ----------
HP_ACC_TOL = 0.0015

# ---------- FINAL ENSEMBLE ----------
FINAL_ENSEMBLE_SEEDS = [42]

# ---------- MODEL CANDIDATES ----------
SEARCH_CANDIDATES = [
    {"embed_dim": 128, "num_heads": 4, "depth": 2, "ff_mult": 2.0, "dropout": 0.08, "token_dropout": 0.01, "lr": 1.0e-3, "weight_decay": 1e-5, "batch_size": 2048},
    {"embed_dim": 160, "num_heads": 4, "depth": 2, "ff_mult": 2.0, "dropout": 0.10, "token_dropout": 0.02, "lr": 8.0e-4, "weight_decay": 1e-5, "batch_size": 2048},
    {"embed_dim": 192, "num_heads": 4, "depth": 3, "ff_mult": 2.0, "dropout": 0.10, "token_dropout": 0.02, "lr": 8.0e-4, "weight_decay": 1e-5, "batch_size": 1024},
]


# ============================================================
# HELPERS
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


def now():
    return time.strftime("%H:%M:%S")


def safe_read_csv(path):
    if path is None or str(path).strip() == "":
        return None
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)


def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def clip_probs(prob, eps=1e-7):
    return np.clip(np.asarray(prob, dtype=np.float64), eps, 1 - eps)


def safe_auc(y_true, prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, prob))


def _threshold_table(y_true, prob, grid):
    rows = []
    prob = clip_probs(prob)

    for t in grid:
        pred = (prob >= t).astype(int)
        acc = accuracy_score(y_true, pred)
        prec = precision_score(y_true, pred, zero_division=0)
        rec = recall_score(y_true, pred, zero_division=0)
        f1v = f1_score(y_true, pred, zero_division=0)
        mcc = matthews_corrcoef(y_true, pred)

        rows.append({
            "threshold": float(t),
            "accuracy": float(acc),
            "precision": float(prec),
            "recall": float(rec),
            "f1": float(f1v),
            "mcc": float(mcc),
        })

    return pd.DataFrame(rows)


def _pick_best_threshold_from_df(df_thr: pd.DataFrame) -> float:
    if df_thr.empty:
        return 0.50

    df_thr = df_thr.sort_values(
        by=["f1", "recall", "accuracy", "mcc", "precision", "threshold"],
        ascending=[False, False, False, False, False, True]
    ).reset_index(drop=True)

    return float(df_thr.loc[0, "threshold"])


def tune_threshold_constrained(
    y_true,
    prob,
    grid=None,
    acc_drop_tol=0.0015,
    min_dynamic_precision=0.970,
    precision_relax=0.012,
):
    if grid is None:
        grid = np.linspace(0.16, 0.60, 177)

    prob = clip_probs(prob)
    thr_df = _threshold_table(y_true, prob, grid)

    best_acc = float(thr_df["accuracy"].max())
    acc_floor = best_acc - float(acc_drop_tol)

    safe_df = thr_df.loc[thr_df["accuracy"] >= acc_floor].copy()
    if safe_df.empty:
        safe_df = thr_df.copy()

    best_acc_df = thr_df.loc[thr_df["accuracy"] >= best_acc - 1e-12].copy()
    best_acc_df = best_acc_df.sort_values(
        by=["f1", "recall", "mcc", "precision", "threshold"],
        ascending=[False, False, False, False, True]
    ).reset_index(drop=True)

    ref_precision = float(best_acc_df.loc[0, "precision"])
    precision_floor = max(float(min_dynamic_precision), ref_precision - float(precision_relax))

    guarded_df = safe_df.loc[safe_df["precision"] >= precision_floor].copy()
    if guarded_df.empty:
        guarded_df = safe_df.copy()

    return _pick_best_threshold_from_df(guarded_df)


def compute_metrics_requested(y_true, prob, threshold=0.5):
    prob = clip_probs(prob)
    pred = (prob >= threshold).astype(int)

    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, pred, zero_division=0)),
        "logloss": float(log_loss(y_true, prob, labels=[0, 1])),
        "roc_auc": safe_auc(y_true, prob),
        "mcc": float(matthews_corrcoef(y_true, pred)),
    }


class FoldPreprocessor:
    def __init__(self):
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()

    def fit(self, X_df):
        X_num = to_numeric_df(X_df)
        X_imp = self.imputer.fit_transform(X_num)
        self.scaler.fit(X_imp)
        return self

    def transform(self, X_df):
        X_num = to_numeric_df(X_df)
        X_imp = self.imputer.transform(X_num)
        X_scaled = self.scaler.transform(X_imp)
        X_scaled = np.clip(X_scaled, -8.0, 8.0).astype(np.float32)
        return X_scaled

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)


def make_loader(X, y=None, batch_size=1024, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    if y is None:
        ds = TensorDataset(X_t)
    else:
        y_t = torch.tensor(y, dtype=torch.float32)
        ds = TensorDataset(X_t, y_t)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )


@torch.no_grad()
def predict_proba_torch(model, X, batch_size=1024):
    model.eval()
    loader = make_loader(X, y=None, batch_size=batch_size, shuffle=False)
    probs = []

    for batch in loader:
        xb = batch[0].to(DEVICE, non_blocking=True)
        logits = model(xb)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(p)

    return np.concatenate(probs).astype(np.float64)


def fit_torch_model(
    model,
    X_train, y_train,
    X_valid, y_valid,
    lr=1e-3,
    weight_decay=1e-5,
    batch_size=1024,
    max_epochs=12,
    patience=3,
    seed=42
):
    set_seed(seed)
    model = model.to(DEVICE)

    train_loader = make_loader(X_train, y_train, batch_size=batch_size, shuffle=True)
    valid_loader = make_loader(X_valid, y_valid, batch_size=max(1024, batch_size), shuffle=False)

    pos_count = max(float(np.sum(np.asarray(y_train) == 1)), 1.0)
    neg_count = max(float(np.sum(np.asarray(y_train) == 0)), 1.0)

    if USE_MILD_POS_WEIGHT:
        pos_weight_value = float(np.clip((neg_count / pos_count) ** POS_WEIGHT_POWER, 1.0, POS_WEIGHT_MAX))
    else:
        pos_weight_value = 1.0

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    steps_per_epoch = max(1, len(train_loader))
    total_steps = max(1, max_epochs * steps_per_epoch)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=total_steps,
        pct_start=0.20,
        div_factor=8.0,
        final_div_factor=80.0,
    )

    use_amp_now = bool(USE_AMP and DEVICE.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp_now)

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = np.inf
    bad_epochs = 0
    min_delta = 5e-4

    for epoch in range(1, max_epochs + 1):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp_now):
                logits = model(xb)
                loss = criterion(logits, yb)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        model.eval()
        val_probs = []
        for xb, _ in valid_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=use_amp_now):
                logits = model(xb)
                p = torch.sigmoid(logits)
            val_probs.append(p.detach().cpu().numpy())

        val_probs = np.concatenate(val_probs).astype(np.float64)
        val_loss = log_loss(y_valid, clip_probs(val_probs), labels=[0, 1])

        if val_loss < best_val_loss - min_delta:
            best_val_loss = float(val_loss)
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    return model


# ============================================================
# DEEP LEARNING BASELINES
# ============================================================

class MLP2(nn.Module):
    def __init__(self, input_dim, dropout=0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class DeepMLP(nn.Module):
    def __init__(self, input_dim, dropout=0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)



class GatedMLP(nn.Module):
    def __init__(self, input_dim, dropout=0.10):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(256, 256)
        self.fc3 = nn.Linear(128, 128)
        self.out = nn.Linear(64, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.norm(x)
        x = F.glu(self.fc1(x), dim=-1)
        x = self.drop(x)
        x = F.glu(self.fc2(x), dim=-1)
        x = self.drop(x)
        x = F.glu(self.fc3(x), dim=-1)
        x = self.drop(x)
        return self.out(x).squeeze(1)

# ============================================================
# MODEL REGISTRY
# ============================================================

ML_MODELS = {
    "GaussianNB": lambda seed: GaussianNB(),
    "ExtraTrees": lambda seed: ExtraTreesClassifier(n_estimators=300, random_state=seed, n_jobs=-1),
    "RandomForest": lambda seed: RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1),
}

DL_MODELS = {
    "GatedMLP": lambda input_dim: GatedMLP(input_dim=input_dim, dropout=0.10),
    "MLP2": lambda input_dim: MLP2(input_dim=input_dim, dropout=0.10),
    "DeepMLP": lambda input_dim: DeepMLP(input_dim=input_dim, dropout=0.10),
}

TR_MODELS = {}

ALL_MODELS = (
    [("Machine Learning", name) for name in ML_MODELS.keys()] +
    [("Deep Learning", name) for name in DL_MODELS.keys()] +
    [("Transformer", name) for name in TR_MODELS.keys()]
)


def is_sklearn_model(model_name):
    return model_name in ML_MODELS

def is_dl_model(model_name):
    return model_name in DL_MODELS

def is_transformer_model(model_name):
    return model_name in TR_MODELS

def build_model(model_name, input_dim, seed):
    if is_sklearn_model(model_name):
        return ML_MODELS[model_name](seed)
    if is_dl_model(model_name):
        return DL_MODELS[model_name](input_dim)
    if is_transformer_model(model_name):
        return TR_MODELS[model_name](input_dim)
    raise ValueError(f"Unknown model: {model_name}")

def fit_model_same_framework(model_name, X_train, y_train, X_valid, y_valid, phase, seed):
    if is_sklearn_model(model_name):
        model = build_model(model_name, input_dim=X_train.shape[1], seed=seed)
        model.fit(X_train, y_train)
        return model

    model = build_model(model_name, input_dim=X_train.shape[1], seed=seed)

    if is_dl_model(model_name):
        lr = 1.0e-3
        weight_decay = 1e-5
        batch_size = 1024
    else:
        lr = 8.0e-4
        weight_decay = 1e-5
        batch_size = 1024

    if phase == "inner":
        max_epochs = MAX_EPOCHS_INNER
        patience = PATIENCE_INNER
    elif phase == "outer":
        max_epochs = MAX_EPOCHS_OUTER
        patience = PATIENCE_OUTER
    elif phase == "final":
        max_epochs = MAX_EPOCHS_FINAL
        patience = PATIENCE_FINAL
    else:
        raise ValueError("phase must be one of: inner, outer, final")

    model = fit_torch_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_valid=X_valid,
        y_valid=y_valid,
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        max_epochs=max_epochs,
        patience=patience,
        seed=seed,
    )
    return model


def predict_model_prob(model_name, fitted_model, X):
    if is_sklearn_model(model_name):
        return fitted_model.predict_proba(X)[:, 1].astype(np.float64)

    if is_dl_model(model_name) or is_transformer_model(model_name):
        return predict_proba_torch(fitted_model, X, batch_size=1024)

    raise ValueError(f"Unknown model: {model_name}")

def inner_cv_for_baseline(X_df, y, model_name, seed=42):
    skf = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=seed)
    oof_prob = np.zeros(len(y), dtype=np.float64)

    for inner_fold, (tr_idx, va_idx) in enumerate(skf.split(X_df, y), start=1):
        X_tr_df = X_df.iloc[tr_idx].reset_index(drop=True)
        X_va_df = X_df.iloc[va_idx].reset_index(drop=True)
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        prep = FoldPreprocessor()
        X_tr = prep.fit_transform(X_tr_df)
        X_va = prep.transform(X_va_df)

        model = fit_model_same_framework(
            model_name=model_name,
            X_train=X_tr,
            y_train=y_tr,
            X_valid=X_va,
            y_valid=y_va,
            phase="inner",
            seed=seed + inner_fold * 77,
        )

        prob = predict_model_prob(model_name, model, X_va)
        oof_prob[va_idx] = prob

    best_thr = tune_threshold_constrained(
        y_true=y,
        prob=oof_prob,
        grid=THRESHOLD_GRID,
        acc_drop_tol=ACC_DROP_TOL,
        min_dynamic_precision=MIN_DYNAMIC_PRECISION,
        precision_relax=PRECISION_RELAX,
    )

    metrics = compute_metrics_requested(y, oof_prob, threshold=best_thr)
    return best_thr, metrics, oof_prob


# ============================================================
# LOAD DATA
# ============================================================

set_seed(SEED)

print("\n" + "=" * 90)
print(f"[{now()}] LOADING DATA")
print("=" * 90)

train_df = safe_read_csv(TRAIN_PATH)
test_df = safe_read_csv(TEST_PATH)

if TARGET_COLUMN not in train_df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found.")

drop_train = list(set(ID_COLUMNS + DROP_COLUMNS + [TARGET_COLUMN]))
X_train_df = train_df.drop(columns=[c for c in drop_train if c in train_df.columns], errors="ignore").copy()
y_raw = train_df[TARGET_COLUMN].copy()

le = LabelEncoder()
y = le.fit_transform(y_raw)

if len(le.classes_) != 2:
    raise ValueError(f"Binary classification only. Found classes: {list(le.classes_)}")

class_names = [str(x) for x in le.classes_]

print(f"Train shape   : {train_df.shape}")
print(f"Feature shape : {X_train_df.shape}")
print(f"Classes       : {class_names}")
print(f"Device        : {DEVICE}")

if test_df is None or TARGET_COLUMN not in test_df.columns:
    raise ValueError("For this baseline comparison, labeled TEST_PATH is required.")

drop_test = list(set(ID_COLUMNS + DROP_COLUMNS + [TARGET_COLUMN]))
X_test_df = test_df.drop(columns=[c for c in drop_test if c in test_df.columns], errors="ignore").copy()
y_test = le.transform(test_df[TARGET_COLUMN])
has_test_labels = True

print(f"Test shape    : {test_df.shape}")
print(f"Test labels   : present")


# ============================================================
# MAIN BASELINE LOOP
# ============================================================

nested_cv_summary_rows = []
test_summary_rows = []
roc_store = {}

for family_name, model_name in ALL_MODELS:
    print("\n" + "=" * 90)
    print(f"[{now()}] BASELINE MODEL: {family_name} | {model_name}")
    print("=" * 90)

    outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED)

    outer_oof_prob = np.zeros(len(y), dtype=np.float64)
    outer_fold_rows = []

    for outer_fold, (dev_idx, hold_idx) in enumerate(outer_cv.split(X_train_df, y), start=1):
        print(f"\n[{now()}] {model_name} | OUTER FOLD {outer_fold}/{OUTER_FOLDS}")

        X_dev_df = X_train_df.iloc[dev_idx].reset_index(drop=True)
        y_dev = y[dev_idx]

        X_hold_df = X_train_df.iloc[hold_idx].reset_index(drop=True)
        y_hold = y[hold_idx]

        best_thr, inner_metrics, inner_oof_prob = inner_cv_for_baseline(
            X_df=X_dev_df,
            y=y_dev,
            model_name=model_name,
            seed=SEED + outer_fold * 100,
        )

        print(
            f"  inner-cv | acc={inner_metrics['accuracy']:.5f} | "
            f"prec={inner_metrics['precision']:.5f} | rec={inner_metrics['recall']:.5f} | "
            f"f1={inner_metrics['f1_score']:.5f} | auc={inner_metrics['roc_auc']:.5f} | "
            f"logloss={inner_metrics['logloss']:.5f} | mcc={inner_metrics['mcc']:.5f} | "
            f"thr={best_thr:.3f}"
        )

        X_tr_df, X_va_df, y_tr, y_va = train_test_split(
            X_dev_df, y_dev,
            test_size=OUTER_DEV_VALID_SIZE,
            random_state=SEED + outer_fold,
            stratify=y_dev
        )

        prep = FoldPreprocessor()
        X_tr = prep.fit_transform(X_tr_df)
        X_va = prep.transform(X_va_df)
        X_hold = prep.transform(X_hold_df)

        model = fit_model_same_framework(
            model_name=model_name,
            X_train=X_tr,
            y_train=y_tr,
            X_valid=X_va,
            y_valid=y_va,
            phase="outer",
            seed=SEED + outer_fold * 1000,
        )

        hold_prob = predict_model_prob(model_name, model, X_hold)
        outer_oof_prob[hold_idx] = hold_prob

        fold_metrics = compute_metrics_requested(y_hold, hold_prob, threshold=best_thr)
        fold_metrics["outer_fold"] = outer_fold
        fold_metrics["threshold"] = float(best_thr)
        outer_fold_rows.append(fold_metrics)

        print(
            f"  outer-hold | acc={fold_metrics['accuracy']:.5f} | "
            f"prec={fold_metrics['precision']:.5f} | rec={fold_metrics['recall']:.5f} | "
            f"f1={fold_metrics['f1_score']:.5f} | auc={fold_metrics['roc_auc']:.5f} | "
            f"logloss={fold_metrics['logloss']:.5f} | mcc={fold_metrics['mcc']:.5f}"
        )

    outer_df = pd.DataFrame(outer_fold_rows)

    global_threshold = tune_threshold_constrained(
        y_true=y,
        prob=outer_oof_prob,
        grid=THRESHOLD_GRID,
        acc_drop_tol=ACC_DROP_TOL,
        min_dynamic_precision=MIN_DYNAMIC_PRECISION,
        precision_relax=PRECISION_RELAX,
    )

    nested_oof_metrics = compute_metrics_requested(y, outer_oof_prob, threshold=global_threshold)

    nested_cv_summary_rows.append({
        "family": family_name,
        "model": model_name,
        "accuracy": nested_oof_metrics["accuracy"],
        "precision": nested_oof_metrics["precision"],
        "recall": nested_oof_metrics["recall"],
        "f1_score": nested_oof_metrics["f1_score"],
        "logloss": nested_oof_metrics["logloss"],
        "roc_auc": nested_oof_metrics["roc_auc"],
        "mcc": nested_oof_metrics["mcc"],
        "threshold": float(global_threshold),
    })

    print("\nNESTED-CV OOF SUMMARY")
    print(pd.DataFrame([nested_oof_metrics]).round(5).to_string(index=False))

    # --------------------------------------------------------
    # FINAL TRAIN -> FINAL VALID -> TEST
    # --------------------------------------------------------
    X_tr_df, X_va_df, y_tr, y_va = train_test_split(
        X_train_df, y,
        test_size=FINAL_VALID_SIZE,
        random_state=SEED,
        stratify=y
    )

    prep = FoldPreprocessor()
    X_tr = prep.fit_transform(X_tr_df)
    X_va = prep.transform(X_va_df)
    X_te = prep.transform(X_test_df)

    final_model = fit_model_same_framework(
        model_name=model_name,
        X_train=X_tr,
        y_train=y_tr,
        X_valid=X_va,
        y_valid=y_va,
        phase="final",
        seed=SEED,
    )

    final_valid_prob = predict_model_prob(model_name, final_model, X_va)
    final_threshold = tune_threshold_constrained(
        y_true=y_va,
        prob=final_valid_prob,
        grid=THRESHOLD_GRID,
        acc_drop_tol=ACC_DROP_TOL,
        min_dynamic_precision=MIN_DYNAMIC_PRECISION,
        precision_relax=PRECISION_RELAX,
    )

    final_test_prob = predict_model_prob(model_name, final_model, X_te)
    final_test_metrics = compute_metrics_requested(y_test, final_test_prob, threshold=final_threshold)

    test_summary_rows.append({
        "family": family_name,
        "model": model_name,
        "accuracy": final_test_metrics["accuracy"],
        "precision": final_test_metrics["precision"],
        "recall": final_test_metrics["recall"],
        "f1_score": final_test_metrics["f1_score"],
        "logloss": final_test_metrics["logloss"],
        "roc_auc": final_test_metrics["roc_auc"],
        "mcc": final_test_metrics["mcc"],
        "threshold": float(final_threshold),
    })

    fpr, tpr, thresholds = roc_curve(y_test, final_test_prob)
    roc_store[model_name] = {
        "family": family_name,
        "y_true": np.asarray(y_test, dtype=int),
        "y_prob": np.asarray(final_test_prob, dtype=np.float64),
        "fpr": np.asarray(fpr, dtype=np.float64),
        "tpr": np.asarray(tpr, dtype=np.float64),
        "thresholds": np.asarray(thresholds, dtype=np.float64),
        "roc_auc": float(roc_auc_score(y_test, final_test_prob)),
    }

    print("\nFINAL TEST SUMMARY")
    print(pd.DataFrame([final_test_metrics]).round(5).to_string(index=False))


# ============================================================
# OUTPUT TABLES
# ============================================================

nested_cv_df = pd.DataFrame(nested_cv_summary_rows)
test_results_df = pd.DataFrame(test_summary_rows)

sort_cols = ["accuracy", "f1_score", "roc_auc", "mcc"]
nested_cv_df = nested_cv_df.sort_values(by=sort_cols, ascending=[False, False, False, False]).reset_index(drop=True)
test_results_df = test_results_df.sort_values(by=sort_cols, ascending=[False, False, False, False]).reset_index(drop=True)

nested_cv_df_display = nested_cv_df.rename(columns={
    "family": "Family",
    "model": "Model",
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1_score": "F1-Score",
    "logloss": "LogLoss",
    "roc_auc": "ROC-AUC",
    "mcc": "MCC",
    "threshold": "Threshold",
})

test_results_df_display = test_results_df.rename(columns={
    "family": "Family",
    "model": "Model",
    "accuracy": "Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "f1_score": "F1-Score",
    "logloss": "LogLoss",
    "roc_auc": "ROC-AUC",
    "mcc": "MCC",
    "threshold": "Threshold",
})

print("\n" + "=" * 90)
print("NESTED-CV OOF RESULTS")
print("=" * 90)
display(nested_cv_df_display.round(5))

print("\n" + "=" * 90)
print("FINAL TEST RESULTS")
print("=" * 90)
display(test_results_df_display.round(5))

print("\n" + "=" * 90)
print("FINAL TEST RESULTS - MACHINE LEARNING")
print("=" * 90)
display(test_results_df_display[test_results_df_display["Family"] == "Machine Learning"].reset_index(drop=True).round(5))

print("\n" + "=" * 90)
print("FINAL TEST RESULTS - DEEP LEARNING")
print("=" * 90)
display(test_results_df_display[test_results_df_display["Family"] == "Deep Learning"].reset_index(drop=True).round(5))



LOADING DATA
Train shape   : (4000, 21)
Feature shape : (4000, 20)
Classes       : ['0', '1']
Device        : cuda
Test shape    : (1000, 21)
Test labels   : present

FINAL TEST RESULTS - BASELINE MODELS
             Family        Model  Accuracy  Precision   Recall  F1-Score  LogLoss  ROC-AUC
   Machine Learning   GaussianNB     0.893    0.70474  0.99606   0.82545  1.27113  0.96246
   Machine Learning   ExtraTrees     0.963    0.89744  0.96457   0.92979  0.08609  0.99498
   Machine Learning RandomForest     0.971    0.94118  0.94488   0.94303  0.08320  0.99583
      Deep Learning     GatedMLP     0.933    0.82131  0.94094   0.87706  0.17310  0.98804
      Deep Learning         MLP2     0.941    0.84946  0.93307   0.88931  0.12536  0.99036
      Deep Learning      DeepMLP     0.945    0.84192  0.96457   0.89908  0.10005  0.99231


In [ ]:

import os
import time
import copy
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    matthews_corrcoef,
    roc_curve,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# ============================================================
# GLOBAL VARIABLES
# ============================================================

# ---------- DATA ----------
TRAIN_PATH = "/content/train_selected.csv"
TEST_PATH  = "/content/test_selected.csv"
TARGET_COLUMN = "lung_cancer_risk"
ID_COLUMNS = []
DROP_COLUMNS = []

# ---------- REPRO ----------
SEED = 42
# ---------- CV ----------
OUTER_FOLDS = 5
INNER_FOLDS = 2

N_TRIALS = 3
MAX_EPOCHS_INNER = 6
MAX_EPOCHS_OUTER = 10
MAX_EPOCHS_FINAL = 12

PATIENCE_INNER = 2
PATIENCE_OUTER = 3
PATIENCE_FINAL = 3

OUTER_DEV_VALID_SIZE = 0.12
FINAL_VALID_SIZE = 0.10

USE_AMP = True
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- THRESHOLD ----------
THRESHOLD_GRID = np.linspace(0.16, 0.60, 177)
ACC_DROP_TOL = 0.0015

# Dynamic precision protection
MIN_DYNAMIC_PRECISION = 0.970
PRECISION_RELAX = 0.012

# ---------- LOSS ----------
USE_MILD_POS_WEIGHT = True
POS_WEIGHT_POWER = 0.35
POS_WEIGHT_MAX = 1.25


# ============================================================
# HELPERS
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


def now():
    return time.strftime("%H:%M:%S")


def safe_read_csv(path):
    if path is None or str(path).strip() == "":
        return None
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)


def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def clip_probs(prob, eps=1e-7):
    return np.clip(np.asarray(prob, dtype=np.float64), eps, 1 - eps)


def safe_auc(y_true, prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, prob))


def _threshold_table(y_true, prob, grid):
    rows = []
    prob = clip_probs(prob)

    for t in grid:
        pred = (prob >= t).astype(int)
        rows.append({
            "threshold": float(t),
            "accuracy": float(accuracy_score(y_true, pred)),
            "precision": float(precision_score(y_true, pred, zero_division=0)),
            "recall": float(recall_score(y_true, pred, zero_division=0)),
            "f1": float(f1_score(y_true, pred, zero_division=0)),
            "mcc": float(matthews_corrcoef(y_true, pred)),
        })

    return pd.DataFrame(rows)


def _pick_best_threshold_from_df(df_thr: pd.DataFrame) -> float:
    if df_thr.empty:
        return 0.50

    df_thr = df_thr.sort_values(
        by=["f1", "recall", "accuracy", "mcc", "precision", "threshold"],
        ascending=[False, False, False, False, False, True]
    ).reset_index(drop=True)

    return float(df_thr.loc[0, "threshold"])


def tune_threshold_constrained(
    y_true,
    prob,
    grid=None,
    acc_drop_tol=0.0015,
    min_dynamic_precision=0.970,
    precision_relax=0.012,
):
    if grid is None:
        grid = np.linspace(0.16, 0.60, 177)

    prob = clip_probs(prob)
    thr_df = _threshold_table(y_true, prob, grid)

    best_acc = float(thr_df["accuracy"].max())
    acc_floor = best_acc - float(acc_drop_tol)

    safe_df = thr_df.loc[thr_df["accuracy"] >= acc_floor].copy()
    if safe_df.empty:
        safe_df = thr_df.copy()

    best_acc_df = thr_df.loc[thr_df["accuracy"] >= best_acc - 1e-12].copy()
    best_acc_df = best_acc_df.sort_values(
        by=["f1", "recall", "mcc", "precision", "threshold"],
        ascending=[False, False, False, False, True]
    ).reset_index(drop=True)

    ref_precision = float(best_acc_df.loc[0, "precision"])
    precision_floor = max(float(min_dynamic_precision), ref_precision - float(precision_relax))

    guarded_df = safe_df.loc[safe_df["precision"] >= precision_floor].copy()
    if guarded_df.empty:
        guarded_df = safe_df.copy()

    return _pick_best_threshold_from_df(guarded_df)


def compute_metrics_requested(y_true, prob, threshold=0.5):
    prob = clip_probs(prob)
    pred = (prob >= threshold).astype(int)

    return {
        "Accuracy": float(accuracy_score(y_true, pred)),
        "Precision": float(precision_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "F1-Score": float(f1_score(y_true, pred, zero_division=0)),
        "LogLoss": float(log_loss(y_true, prob, labels=[0, 1])),
        "ROC-AUC": safe_auc(y_true, prob),
        "MCC": float(matthews_corrcoef(y_true, pred)),
    }


# ============================================================
# PREPROCESSOR
# ============================================================

class FoldPreprocessor:
    def __init__(self):
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()

    def fit(self, X_df):
        X_num = to_numeric_df(X_df)
        X_imp = self.imputer.fit_transform(X_num)
        self.scaler.fit(X_imp)
        return self

    def transform(self, X_df):
        X_num = to_numeric_df(X_df)
        X_imp = self.imputer.transform(X_num)
        X_scaled = self.scaler.transform(X_imp)
        X_scaled = np.clip(X_scaled, -8.0, 8.0).astype(np.float32)
        return X_scaled

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)


# ============================================================
# TORCH DATALOADER + TRAIN
# ============================================================

def make_loader(X, y=None, batch_size=1024, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    if y is None:
        ds = TensorDataset(X_t)
    else:
        y_t = torch.tensor(y, dtype=torch.float32)
        ds = TensorDataset(X_t, y_t)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )


@torch.no_grad()
def predict_proba_torch(model, X, batch_size=1024):
    model.eval()
    loader = make_loader(X, y=None, batch_size=batch_size, shuffle=False)
    probs = []

    for batch in loader:
        xb = batch[0].to(DEVICE, non_blocking=True)
        logits = model(xb)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(p)

    return np.concatenate(probs).astype(np.float64)


def fit_torch_model(
    model,
    X_train, y_train,
    X_valid, y_valid,
    lr=8.0e-4,
    weight_decay=1e-5,
    batch_size=1024,
    max_epochs=12,
    patience=3,
    seed=42
):
    set_seed(seed)
    model = model.to(DEVICE)

    train_loader = make_loader(X_train, y_train, batch_size=batch_size, shuffle=True)
    valid_loader = make_loader(X_valid, y_valid, batch_size=max(1024, batch_size), shuffle=False)

    pos_count = max(float(np.sum(np.asarray(y_train) == 1)), 1.0)
    neg_count = max(float(np.sum(np.asarray(y_train) == 0)), 1.0)

    if USE_MILD_POS_WEIGHT:
        pos_weight_value = float(np.clip((neg_count / pos_count) ** POS_WEIGHT_POWER, 1.0, POS_WEIGHT_MAX))
    else:
        pos_weight_value = 1.0

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    steps_per_epoch = max(1, len(train_loader))
    total_steps = max(1, max_epochs * steps_per_epoch)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=total_steps,
        pct_start=0.20,
        div_factor=8.0,
        final_div_factor=80.0,
    )

    use_amp_now = bool(USE_AMP and DEVICE.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp_now)

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = np.inf
    bad_epochs = 0
    min_delta = 5e-4

    for epoch in range(1, max_epochs + 1):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp_now):
                logits = model(xb)
                loss = criterion(logits, yb)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        model.eval()
        val_probs = []
        for xb, _ in valid_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=use_amp_now):
                logits = model(xb)
                p = torch.sigmoid(logits)
            val_probs.append(p.detach().cpu().numpy())

        val_probs = np.concatenate(val_probs).astype(np.float64)
        val_loss = log_loss(y_valid, clip_probs(val_probs), labels=[0, 1])

        if val_loss < best_val_loss - min_delta:
            best_val_loss = float(val_loss)
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    return model


# ============================================================
# COMMON TRANSFORMER BLOCKS
# ============================================================

class FeatureTokenizer(nn.Module):
    def __init__(self, n_features, embed_dim, token_dropout=0.01):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, embed_dim) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, embed_dim))
        self.feature_embed = nn.Parameter(torch.randn(n_features, embed_dim) * 0.02)
        self.norm = nn.LayerNorm(embed_dim)
        self.token_dropout = float(token_dropout)

    def forward(self, x):
        tok = x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0) + self.feature_embed.unsqueeze(0)
        if self.training and self.token_dropout > 0:
            keep = (torch.rand((x.size(0), x.size(1), 1), device=x.device) > self.token_dropout).float()
            tok = tok * keep
        return self.norm(tok)


class MHABlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_mult=2.0, dropout=0.10):
        super().__init__()
        ff_dim = int(embed_dim * ff_mult)

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff1 = nn.Linear(embed_dim, ff_dim * 2)
        self.ff2 = nn.Linear(ff_dim, embed_dim)
        self.drop2 = nn.Dropout(dropout)

        self.gate = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False)
        x = x + self.drop1(attn_out)

        y = self.norm2(x)
        a, b = self.ff1(y).chunk(2, dim=-1)
        y = a * F.gelu(b)
        y = self.ff2(self.drop2(y))

        gate = self.gate(x)
        x = x + gate * y
        return x


class SimpleFFN(nn.Module):
    def __init__(self, dim, ff_mult=2.0, dropout=0.10):
        super().__init__()
        ff_dim = int(dim * ff_mult)
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.net(x)


class PoolHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.10):
        super().__init__()
        hidden2 = max(hidden_dim // 2, 64)
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

# ============================================================
#  TRANSFORMER BASELINES
# ============================================================

# 1) TabTransformer
class TabTransformerModel(nn.Module):
    def __init__(self, input_dim, embed_dim=128, num_heads=4, depth=2, ff_mult=2.0, dropout=0.10, token_dropout=0.01):
        super().__init__()
        self.tokenizer = FeatureTokenizer(input_dim, embed_dim, token_dropout)
        self.blocks = nn.ModuleList([MHABlock(embed_dim, num_heads, ff_mult, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = PoolHead(embed_dim, hidden_dim=max(embed_dim, 128), dropout=dropout)

    def forward(self, x):
        tok = self.tokenizer(x)
        for blk in self.blocks:
            tok = blk(tok)
        tok = self.norm(tok)
        rep = tok.mean(dim=1)
        return self.head(rep)


# 2) SAINT-inspired
class SAINTBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_mult=2.0, dropout=0.10):
        super().__init__()
        self.col_block = MHABlock(embed_dim, num_heads, ff_mult, dropout)

        self.row_norm = nn.LayerNorm(embed_dim)
        self.row_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.row_drop = nn.Dropout(dropout)

        self.ff = SimpleFFN(embed_dim, ff_mult, dropout)

    def forward(self, x):
        x = self.col_block(x)

        # row attention using sample summaries
        row_repr = x.mean(dim=1)             # [B, E]
        row_seq = row_repr.unsqueeze(0)      # [1, B, E]
        row_seq_n = self.row_norm(row_seq)
        row_out, _ = self.row_attn(row_seq_n, row_seq_n, row_seq_n, need_weights=False)
        row_out = row_seq + self.row_drop(row_out)
        row_out = row_out.squeeze(0).unsqueeze(1)   # [B, 1, E]

        x = x + row_out
        x = self.ff(x)
        return x


class SAINTModel(nn.Module):
    def __init__(self, input_dim, embed_dim=128, num_heads=4, depth=2, ff_mult=2.0, dropout=0.10, token_dropout=0.01):
        super().__init__()
        self.tokenizer = FeatureTokenizer(input_dim, embed_dim, token_dropout)
        self.blocks = nn.ModuleList([SAINTBlock(embed_dim, num_heads, ff_mult, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = PoolHead(embed_dim, hidden_dim=max(embed_dim, 128), dropout=dropout)

    def forward(self, x):
        tok = self.tokenizer(x)
        for blk in self.blocks:
            tok = blk(tok)
        tok = self.norm(tok)
        rep = tok.mean(dim=1)
        return self.head(rep)


# 3) AutoInt-inspired
class AutoIntLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.10):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        y = self.norm(x)
        out, _ = self.attn(y, y, y, need_weights=False)
        out = self.proj(out)
        return F.relu(x + self.drop(out))


class AutoIntModel(nn.Module):
    def __init__(self, input_dim, embed_dim=128, num_heads=4, depth=3, dropout=0.10, token_dropout=0.01):
        super().__init__()
        self.tokenizer = FeatureTokenizer(input_dim, embed_dim, token_dropout)
        self.layers = nn.ModuleList([AutoIntLayer(embed_dim, num_heads, dropout) for _ in range(depth)])
        self.head = PoolHead(embed_dim, hidden_dim=max(embed_dim, 128), dropout=dropout)

    def forward(self, x):
        tok = self.tokenizer(x)
        for lyr in self.layers:
            tok = lyr(tok)
        rep = tok.mean(dim=1)
        return self.head(rep)



NumericFeatureTokenizer = FeatureTokenizer
TransformerBlock = MHABlock

class TabFGT(nn.Module):
    def __init__(
        self,
        input_dim,
        embed_dim=160,
        num_heads=4,
        depth=2,
        ff_mult=2.0,
        dropout=0.10,
        token_dropout=0.01,
    ):
        super().__init__()

        self.tokenizer = NumericFeatureTokenizer(
            n_features=input_dim,
            embed_dim=embed_dim,
            token_dropout=token_dropout
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.cls_bias = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.blocks = nn.ModuleList([
            TransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ff_mult=ff_mult,
                dropout=dropout
            )
            for _ in range(depth)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)

        head_dim = embed_dim * 2
        hidden2 = max(embed_dim, 128)

        self.head = nn.Sequential(
            nn.LayerNorm(head_dim),
            nn.Linear(head_dim, hidden2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, hidden2 // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2 // 2, 1),
        )

    def forward(self, x):
        tok = self.tokenizer(x)
        cls = self.cls_token.expand(x.size(0), -1, -1) + self.cls_bias
        tok = torch.cat([cls, tok], dim=1)

        for block in self.blocks:
            tok = block(tok)

        tok = self.final_norm(tok)

        cls_tok = tok[:, 0]
        feat_tok = tok[:, 1:]
        mean_tok = feat_tok.mean(dim=1)

        rep = torch.cat([cls_tok, mean_tok], dim=1)
        out = self.head(rep).squeeze(1)
        return out

# ============================================================
# MODEL REGISTRY
# ============================================================

EXTRA_TR_MODELS = {
    "SAINT": lambda input_dim: SAINTModel(input_dim=input_dim, embed_dim=128, num_heads=4, depth=2, ff_mult=2.0, dropout=0.10, token_dropout=0.01),
    "AutoInt": lambda input_dim: AutoIntModel(input_dim=input_dim, embed_dim=128, num_heads=4, depth=3, dropout=0.10, token_dropout=0.01),
    "TabTransformer": lambda input_dim: TabTransformerModel(input_dim=input_dim, embed_dim=128, num_heads=4, depth=2, ff_mult=2.0, dropout=0.10, token_dropout=0.01),
    "TabFGT": lambda input_dim: TabFGT(input_dim=input_dim, embed_dim=192, num_heads=4, depth=3, ff_mult=2.0, dropout=0.10, token_dropout=0.02),
}

def fit_extra_transformer_model(model_name, X_train, y_train, X_valid, y_valid, phase, seed):
    model = EXTRA_TR_MODELS[model_name](X_train.shape[1])

    if phase == "inner":
        max_epochs = MAX_EPOCHS_INNER
        patience = PATIENCE_INNER
    elif phase == "outer":
        max_epochs = MAX_EPOCHS_OUTER
        patience = PATIENCE_OUTER
    elif phase == "final":
        max_epochs = MAX_EPOCHS_FINAL
        patience = PATIENCE_FINAL
    else:
        raise ValueError("phase must be inner / outer / final")

    model = fit_torch_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_valid=X_valid,
        y_valid=y_valid,
        lr=8.0e-4,
        weight_decay=1e-5,
        batch_size=1024,
        max_epochs=max_epochs,
        patience=patience,
        seed=seed,
    )
    return model

def inner_cv_for_extra_transformer(X_df, y_arr, model_name, seed=42):
    skf = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=seed)
    oof_prob = np.zeros(len(y_arr), dtype=np.float64)

    for inner_fold, (tr_idx, va_idx) in enumerate(skf.split(X_df, y_arr), start=1):
        X_tr_df = X_df.iloc[tr_idx].reset_index(drop=True)
        X_va_df = X_df.iloc[va_idx].reset_index(drop=True)
        y_tr = y_arr[tr_idx]
        y_va = y_arr[va_idx]

        prep = FoldPreprocessor()
        X_tr = prep.fit_transform(X_tr_df)
        X_va = prep.transform(X_va_df)

        model = fit_extra_transformer_model(
            model_name=model_name,
            X_train=X_tr,
            y_train=y_tr,
            X_valid=X_va,
            y_valid=y_va,
            phase="inner",
            seed=seed + inner_fold * 77,
        )

        prob = predict_proba_torch(model, X_va, batch_size=1024)
        oof_prob[va_idx] = prob

    best_thr = tune_threshold_constrained(
        y_true=y_arr,
        prob=oof_prob,
        grid=THRESHOLD_GRID,
        acc_drop_tol=ACC_DROP_TOL,
        min_dynamic_precision=MIN_DYNAMIC_PRECISION,
        precision_relax=PRECISION_RELAX,
    )
    metrics = compute_metrics_requested(y_arr, oof_prob, threshold=best_thr)
    return best_thr, metrics, oof_prob

# ============================================================
# LOAD DATA
# ============================================================

set_seed(SEED)

print("\n" + "=" * 90)
print(f"[{now()}] LOADING DATA")
print("=" * 90)

train_df = safe_read_csv(TRAIN_PATH)
test_df = safe_read_csv(TEST_PATH)

if TARGET_COLUMN not in train_df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found.")

if test_df is None or TARGET_COLUMN not in test_df.columns:
    raise ValueError("For this code, labeled TEST_PATH is required.")

drop_train = list(set(ID_COLUMNS + DROP_COLUMNS + [TARGET_COLUMN]))
drop_test = list(set(ID_COLUMNS + DROP_COLUMNS + [TARGET_COLUMN]))

X_train_df = train_df.drop(columns=[c for c in drop_train if c in train_df.columns], errors="ignore").copy()
X_test_df = test_df.drop(columns=[c for c in drop_test if c in test_df.columns], errors="ignore").copy()

le = LabelEncoder()
y = le.fit_transform(train_df[TARGET_COLUMN].copy())
y_test = le.transform(test_df[TARGET_COLUMN].copy())

if len(le.classes_) != 2:
    raise ValueError(f"Binary classification only. Found classes: {list(le.classes_)}")

class_names = [str(x) for x in le.classes_]

print(f"Train shape   : {train_df.shape}")
print(f"Test shape    : {test_df.shape}")
print(f"Feature shape : {X_train_df.shape}")
print(f"Classes       : {class_names}")
print(f"Device        : {DEVICE}")


# ============================================================
# RUN TRANSFORMER BASELINES
# ============================================================

extra_transformer_nested_rows = []
extra_transformer_test_rows = []
extra_transformer_roc_store = {}

for model_name in EXTRA_TR_MODELS.keys():
    print("\n" + "=" * 90)
    print(f"[{now()}] ADDITIONAL TRANSFORMER BASELINE: {model_name}")
    print("=" * 90)

    outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED)
    outer_oof_prob = np.zeros(len(y), dtype=np.float64)

    for outer_fold, (dev_idx, hold_idx) in enumerate(outer_cv.split(X_train_df, y), start=1):
        print(f"\n[{now()}] {model_name} | OUTER FOLD {outer_fold}/{OUTER_FOLDS}")

        X_dev_df = X_train_df.iloc[dev_idx].reset_index(drop=True)
        y_dev = y[dev_idx]

        X_hold_df = X_train_df.iloc[hold_idx].reset_index(drop=True)
        y_hold = y[hold_idx]

        best_thr, inner_metrics, _ = inner_cv_for_extra_transformer(
            X_df=X_dev_df,
            y_arr=y_dev,
            model_name=model_name,
            seed=SEED + outer_fold * 100,
        )

        print(
            f"  inner-cv | acc={inner_metrics['Accuracy']:.5f} | "
            f"prec={inner_metrics['Precision']:.5f} | rec={inner_metrics['Recall']:.5f} | "
            f"f1={inner_metrics['F1-Score']:.5f} | auc={inner_metrics['ROC-AUC']:.5f} | "
            f"logloss={inner_metrics['LogLoss']:.5f} | mcc={inner_metrics['MCC']:.5f} | "
            f"thr={best_thr:.3f}"
        )

        X_tr_df, X_va_df, y_tr, y_va = train_test_split(
            X_dev_df, y_dev,
            test_size=OUTER_DEV_VALID_SIZE,
            random_state=SEED + outer_fold,
            stratify=y_dev
        )

        prep = FoldPreprocessor()
        X_tr = prep.fit_transform(X_tr_df)
        X_va = prep.transform(X_va_df)
        X_hold = prep.transform(X_hold_df)

        model = fit_extra_transformer_model(
            model_name=model_name,
            X_train=X_tr,
            y_train=y_tr,
            X_valid=X_va,
            y_valid=y_va,
            phase="outer",
            seed=SEED + outer_fold * 1000,
        )

        hold_prob = predict_proba_torch(model, X_hold, batch_size=1024)
        outer_oof_prob[hold_idx] = hold_prob

        fold_metrics = compute_metrics_requested(y_hold, hold_prob, threshold=best_thr)

        print(
            f"  outer-hold | acc={fold_metrics['Accuracy']:.5f} | "
            f"prec={fold_metrics['Precision']:.5f} | rec={fold_metrics['Recall']:.5f} | "
            f"f1={fold_metrics['F1-Score']:.5f} | auc={fold_metrics['ROC-AUC']:.5f} | "
            f"logloss={fold_metrics['LogLoss']:.5f} | mcc={fold_metrics['MCC']:.5f}"
        )

    global_threshold = tune_threshold_constrained(
        y_true=y,
        prob=outer_oof_prob,
        grid=THRESHOLD_GRID,
        acc_drop_tol=ACC_DROP_TOL,
        min_dynamic_precision=MIN_DYNAMIC_PRECISION,
        precision_relax=PRECISION_RELAX,
    )

    nested_oof_metrics = compute_metrics_requested(y, outer_oof_prob, threshold=global_threshold)

    extra_transformer_nested_rows.append({
        "Family": "Transformer",
        "Model": model_name,
        "Accuracy": nested_oof_metrics["Accuracy"],
        "Precision": nested_oof_metrics["Precision"],
        "Recall": nested_oof_metrics["Recall"],
        "F1-Score": nested_oof_metrics["F1-Score"],
        "LogLoss": nested_oof_metrics["LogLoss"],
        "ROC-AUC": nested_oof_metrics["ROC-AUC"],
        "MCC": nested_oof_metrics["MCC"],
        "Threshold": float(global_threshold),
    })

    print("\nNESTED-CV OOF SUMMARY")
    print(pd.DataFrame([nested_oof_metrics]).round(5).to_string(index=False))

    # --------------------------------------------------------
    # FINAL TRAIN -> FINAL VALID -> FINAL TEST
    # --------------------------------------------------------
    X_tr_df, X_va_df, y_tr, y_va = train_test_split(
        X_train_df, y,
        test_size=FINAL_VALID_SIZE,
        random_state=SEED,
        stratify=y
    )

    prep = FoldPreprocessor()
    X_tr = prep.fit_transform(X_tr_df)
    X_va = prep.transform(X_va_df)
    X_te = prep.transform(X_test_df)

    final_model = fit_extra_transformer_model(
        model_name=model_name,
        X_train=X_tr,
        y_train=y_tr,
        X_valid=X_va,
        y_valid=y_va,
        phase="final",
        seed=SEED,
    )

    final_valid_prob = predict_proba_torch(final_model, X_va, batch_size=1024)
    final_threshold = tune_threshold_constrained(
        y_true=y_va,
        prob=final_valid_prob,
        grid=THRESHOLD_GRID,
        acc_drop_tol=ACC_DROP_TOL,
        min_dynamic_precision=MIN_DYNAMIC_PRECISION,
        precision_relax=PRECISION_RELAX,
    )

    final_test_prob = predict_proba_torch(final_model, X_te, batch_size=1024)
    final_test_metrics = compute_metrics_requested(y_test, final_test_prob, threshold=final_threshold)

    extra_transformer_test_rows.append({
        "Family": "Transformer",
        "Model": model_name,
        "Accuracy": final_test_metrics["Accuracy"],
        "Precision": final_test_metrics["Precision"],
        "Recall": final_test_metrics["Recall"],
        "F1-Score": final_test_metrics["F1-Score"],
        "LogLoss": final_test_metrics["LogLoss"],
        "ROC-AUC": final_test_metrics["ROC-AUC"],
        "MCC": final_test_metrics["MCC"],
        "Threshold": float(final_threshold),
    })

    fpr, tpr, thresholds = roc_curve(y_test, final_test_prob)
    extra_transformer_roc_store[model_name] = {
        "Family": "Transformer",
        "Model": model_name,
        "y_true": np.asarray(y_test, dtype=int),
        "y_prob": np.asarray(final_test_prob, dtype=np.float64),
        "fpr": np.asarray(fpr, dtype=np.float64),
        "tpr": np.asarray(tpr, dtype=np.float64),
        "thresholds": np.asarray(thresholds, dtype=np.float64),
        "roc_auc": float(roc_auc_score(y_test, final_test_prob)),
    }

    print("\nFINAL TEST SUMMARY")
    print(pd.DataFrame([final_test_metrics]).round(5).to_string(index=False))


# ============================================================
# DISPLAY RESULTS
# ============================================================

extra_transformer_nested_df_display = pd.DataFrame(extra_transformer_nested_rows).sort_values(
    by=["Accuracy", "F1-Score", "ROC-AUC", "MCC"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

extra_transformer_test_df_display = pd.DataFrame(extra_transformer_test_rows).sort_values(
    by=["Accuracy", "F1-Score", "ROC-AUC", "MCC"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

print("\n" + "=" * 90)
print("TRANSFORMER BASELINES — NESTED-CV OOF RESULTS")
print("=" * 90)
display(extra_transformer_nested_df_display.round(5))

print("\n" + "=" * 90)
print("TRANSFORMER BASELINES — FINAL TEST RESULTS")
print("=" * 90)
display(extra_transformer_test_df_display.round(5))



LOADING DATA
Train shape   : (4000, 21)
Test shape    : (1000, 21)
Feature shape : (4000, 20)
Classes       : ['0', '1']
Device        : cuda

TRANSFORMER BASELINES - FINAL TEST RESULTS
      Family          Model  Accuracy  Precision   Recall  F1-Score  LogLoss  ROC-AUC
 Transformer          SAINT     0.974    0.95968  0.93701   0.94821  0.06732  0.99772
 Transformer        AutoInt     0.975    0.95984  0.94094   0.95030  0.06241  0.99765
 Transformer TabTransformer     0.977    0.95652  0.95276   0.95464  0.05883  0.99788
 Transformer         TabFGT     0.980    0.96800  0.95276   0.96032  0.05014  0.99816
